# W1D6：周项目——电商订单分析工具 + Git实战

**今天不学新概念。** 把D1-D5学的全部武器拿出来，独立完成一个能写进简历的项目。

---

## 📋 项目背景

你是速卖通（AliExpress）的数据分析实习生。这一周你从零开始学了Python：

| 天 | 学的 | 在真实工作里用来干嘛 |
|----|------|-------------------|
| D1 | 容器(list/dict/set/tuple) | 存数据——订单list套dict、统计dict、去重set |
| D2 | 控制流(if/for/while/try) | 判断大单小单、遍历订单、异常不崩溃 |
| D3 | 函数(def/lambda/推导式) | 封装分析逻辑——一个函数一个分析维度 |
| D4 | 文件IO+字符串+正则 | 读CSV、清洗脏数据、格式化报告 |
| D5 | 模块化+项目结构 | 把代码拆成io/clean/analyze，团队能复用 |

周五下午，mentor小王把你叫到屏幕前：

> "这一周每个技能单拿出来你都会了。但数据分析师不是靠单个技能吃饭的——是把它们**串起来**，从原始数据一直做到可交付的报告。"
> 
> "我把上周的三个CSV给你：`orders.csv`（订单表）、`users.csv`（用户表）、`products.csv`（商品表）。你用纯Python给我出一份**销售日报**，四个维度：总览、城市对比、品类分析、大单分析。再加3条能落地的业务建议。"
> 
> "下午三点运营开会要用。另外——代码放GitHub上，这是团队的标配。能搞定不？"

---

### ⚠️ 真实数据不会干净——你要处理的问题

mentor给你的CSV里故意藏了三个坑（跟真实工作一模一样）：
- **金额是"待定"** → 这条订单跳过，但打印警告
- **金额写成"45元"** → 用正则/字符串处理提取数字
- **城市名前后有空格**（" 北京 "） → 清洗时strip

这些不是bug——是真实数据里的日常。D4学的字符串方法和正则就是用来对付它们的。

---

### 🗺️ 你今天要调用的全部武器

| 来自 | 武器 | 在这个项目中怎么用 |
|------|------|------------------|
| D1 | list/dict/tuple | 订单list套dict、城市统计dict、品类统计dict、list推导式 |
| D2 | for/if/try/except | 遍历订单、条件判断（大单threshold）、异常处理（金额转换失败） |
| D3 | def/return/推导式 | 封装overview()/by_city()/by_category()/big_orders() 四个分析函数 |
| D4 | csv.DictReader/f-string/re.sub | 读CSV、正则提取金额数字、f-string格式化报告行 |
| D5 | import/if __name__ | 拆成src/io.py + clean.py + analyze.py + main.py |
| D6 | Git | init/add/commit/push + README.md |

---

## 🎯 你的5个交付物

| # | 交付物 | 用到的武器 | 验收标准 |
|----|--------|---------|----------|
| ① | `src/io.py` | D4文件IO、D5模块化 | `read_orders()`返回list[dict]，金额是int，脏数据有警告不崩溃 |
| ② | `src/clean.py` | D1容器、D2控制流、D4字符串/正则 | 缺失值跳过、"45元"→45、" 北京 "→"北京" |
| ③ | `src/analyze.py` | D2 for/if、D3函数、D1容器 | 四个分析函数：overview/by_city/by_category/big_orders |
| ④ | `main.py` + 报告输出 | D5代码组织、D4 f-string、D1-D3全部 | `python main.py` 一键输出完整销售日报 |
| ⑤ | Git仓库 + README | Git操作、Markdown | `git push` 到GitHub，README让别人能复现 |

**报告必须包含的五个部分：**
1. 总览：总订单数、总销售额、平均单额、最高/最低单额（含商品名）
2. 城市对比：每个城市的订单数和销售额（按销售额降序排列）
3. 品类分析：每个品类的订单数、销售额、占总销售额百分比
4. 大单分析：大单（>500元）数量和占比、大单商品列表、大单vs小单均价对比
5. 3条可执行的业务建议（具体到品类/城市级别，每条有数据支撑）

---

**开始之前深呼吸。逐个交付物做，完成一个再想下一个。不用一次想清楚全部。**

In [ ]:
# ============================================================
# 【STEP 0】创建项目骨架 + CSV数据文件
# ============================================================
# 这一步帮你搭好目录结构和原始数据。
# 仔细读每一行注释——它们告诉你每个文件的用途。

import os

# --- 0.1 创建目录结构 ---
os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)
os.makedirs('output', exist_ok=True)
print('✅ 目录结构创建完毕：data/  src/  output/')

# --- 0.2 创建 src/__init__.py（让 src 成为一个 Python 包） ---
with open('src/__init__.py', 'w', encoding='utf-8') as f:
    f.write('# src 包——电商订单分析工具\n')
print('✅ src/__init__.py 创建完毕')

# --- 0.3 创建 orders.csv（15条订单，含3个脏数据） ---
# ⚠️ 注意 D010 金额="待定"、D013 金额="45元"、D015 城市=" 北京 "
orders_csv = '''订单号,商品,品类,金额,城市,日期
D001,蓝牙耳机,数码,299,北京,2024-01-08
D002,机械键盘,数码,1599,上海,2024-01-08
D003,手机壳,数码,89,北京,2024-01-09
D004,充电宝,数码,499,深圳,2024-01-09
D005,显示器,数码,2599,上海,2024-01-10
D006,瑜伽垫,运动,129,北京,2024-01-10
D007,跑步鞋,运动,699,深圳,2024-01-11
D008,运动水壶,运动,59,上海,2024-01-11
D009,跳绳,运动,35,北京,2024-01-12
D010,咖啡豆,食品,待定,深圳,2024-01-12
D011,坚果礼盒,食品,199,上海,2024-01-13
D012,茶叶,食品,299,北京,2024-01-13
D013,蜂蜜,食品,45元,深圳,2024-01-14
D014,枸杞,食品,69,上海,2024-01-14
D015,蛋白粉,运动,399, 北京 ,2024-01-14'''

with open('data/orders.csv', 'w', encoding='utf-8') as f:
    f.write(orders_csv)
print('✅ data/orders.csv 创建完毕（15条，含3个脏数据）')

# --- 0.4 创建 users.csv（8个用户） ---
users_csv = '''用户ID,姓名,城市,注册日期,会员等级
U01,张三,北京,2023-06-15,金牌
U02,李四,上海,2023-08-20,银牌
U03,王五,深圳,2023-09-01,金牌
U04,赵六,北京,2024-01-02,普通
U05,陈七,上海,2023-12-10,银牌
U06,周八,深圳,2023-07-25,普通
U07,吴九,北京,2023-11-08,银牌
U08,郑十,上海,2023-05-30,金牌'''

with open('data/users.csv', 'w', encoding='utf-8') as f:
    f.write(users_csv)
print('✅ data/users.csv 创建完毕（8个用户）')

# --- 0.5 创建 products.csv（8个商品） ---
products_csv = '''商品ID,名称,品类,成本,售价
P01,蓝牙耳机,数码,150,299
P02,机械键盘,数码,800,1599
P03,手机壳,数码,20,89
P04,显示器,数码,1500,2599
P05,跑步鞋,运动,350,699
P06,坚果礼盒,食品,80,199
P07,茶叶,食品,120,299
P08,蛋白粉,运动,180,399'''

with open('data/products.csv', 'w', encoding='utf-8') as f:
    f.write(products_csv)
print('✅ data/products.csv 创建完毕（8个商品）')

print('\n📁 项目骨架已就绪，开始写交付物吧！')

In [ ]:
# ============================================================
# 【交付物①】src/io.py —— 把三份CSV读进来
# ============================================================
# 用到的武器：D4-csv.DictReader、D4-open/with、D2-try/except
# 
# 小王："第一步最简单也最容易翻车——把数据读进来。
#        CSV读进来的数字是字符串！忘了转换后面全白做。"
# 
# 任务：写三个函数，每个函数读一个CSV → 返回list[dict]
#   - read_orders(path)   → 金额必须转成int，转不了就跳过+警告
#   - read_users(path)    → 直接读，字段都是字符串
#   - read_products(path) → 成本和售价必须转成int
# 
# 验收标准：
#   ✓ 读取 data/orders.csv → 返回14条（脏数据那条自动跳过）
#   ✓ 每条订单的"金额"字段是int类型（不是"299"字符串）
#   ✓ 遇到"待定"打印警告但不崩溃
# 
# 👇 在下面写你的 src/io.py 代码：

import csv

# --- read_orders() ---



# --- read_users() ---



# --- read_products() ---



# --- 测试你的函数 ---
# orders = read_orders('data/orders.csv')
# users = read_users('data/users.csv')
# products = read_products('data/products.csv')
# print(f"订单:{len(orders)}条 | 用户:{len(users)}个 | 商品:{len(products)}个")
# print(f"第1条订单金额类型: {type(orders[0]['金额'])}")  # 应该是 <class 'int'>
# print(f"第1条订单: {orders[0]}")

In [ ]:
# ============================================================
# 【交付物②】src/clean.py —— 清洗脏数据
# ============================================================
# 用到的武器：D4-str.strip/正则、D2-for、D1-list/dict
# 
# 小王："io.py 已经把'待定'这种读不进来的跳过了。
#        但 '45元' 能读进来，它还是脏的——得洗干净。"
# 
# 任务：写一个 clean_orders() 函数
#   1. 遍历所有订单
#   2. 金额字段：如果还是字符串（如"45元"），用正则提取数字后转int
#      提示：import re; re.sub(r'[^0-9]', '', str_val)
#   3. 城市字段：去掉前后空格（.strip()）
#   4. 可选：检查必填字段（订单号/商品/品类/金额/城市）不缺失
#   5. 返回干净的订单list[dict]
# 
# 验收标准：
#   ✓ "45元" → 45（int）
#   ✓ " 北京 " → "北京"
#   ✓ 清洗后14条订单，每条金额都是int
# 
# 👇 在下面写你的 src/clean.py 代码：

import re
# from src.io import read_orders  # 取消注释来用上一步的io模块

# --- clean_orders() ---
def clean_orders(orders):
    """清洗订单数据：修复金额格式、去掉城市空格、检查必填字段"""
    
    
    
    
    return cleaned

# --- 测试 ---
# raw = read_orders('data/orders.csv')
# clean = clean_orders(raw)
# print(f"清洗后: {len(clean)}条")
# for o in clean:
#     assert isinstance(o['金额'], int), f"{o['订单号']}金额不是int: {type(o['金额'])}"
#     assert o['城市'] == o['城市'].strip(), f"{o['订单号']}城市有空格"
# print("✅ 全部通过验收")

In [ ]:
# ============================================================
# 【交付物③】src/analyze.py —— 四个维度的分析
# ============================================================
# 用到的武器：D2-for+if（核心）、D3-def（封装）、D1-list/dict（数据结构）
# 
# 小王："数据分析的核心就四个字：拆开来看。
#        总共多少→按城市拆→按品类拆→找异常的大单。
#        每个维度一个函数，每个函数30行以内。"
# 
# 任务：写四个分析函数，每个独立可测试
# 
# ① overview(orders) → dict
#    总订单数、总销售额、平均（2位小数）、最高单笔（含商品名）、最低单笔（含商品名）
#    ⚠️ 挑战：最高/最低手动用for找——不调max()/min()！训练底层感觉
#    返回：{"count":14, "total":7018, "avg":501.29, 
#           "max":{"amount":2599, "item":"显示器"}, 
#           "min":{"amount":35, "item":"跳绳"}}
# 
# ② analyze_by_city(orders) → list[dict]（按销售额降序）
#    每个城市的：订单数、销售额、平均每单
#    返回：[{"city":"上海", "count":5, "total":4525, "avg":905.00}, ...]
# 
# ③ analyze_by_category(orders) → list[dict]（按销售额降序）
#    每个品类的：订单数、销售额、占总销售额百分比
#    返回：[{"category":"数码", "count":5, "total":5085, "pct":72.5}, ...]
# 
# ④ analyze_big_orders(orders, threshold=500) → dict
#    大单(>500)的数量、金额、笔数占比、金额占比
#    大单商品列表、大单城市分布、大单均价 vs 小单均价
#    返回：{"big_count":3, "big_total":4897, "count_pct":21.4, 
#           "amount_pct":69.8, "big_items":["机械键盘","显示器","跑步鞋"],
#           "big_avg":1632.33, "small_avg":192.82}
# 
# 👇 在下面写你的四个分析函数（每个函数先写docstring描述它做什么）：

# from src.clean import clean_orders

# --- ① overview ---



# --- ② analyze_by_city ---



# --- ③ analyze_by_category ---



# --- ④ analyze_big_orders ---



# --- 测试全部四个函数 ---
# orders = clean_orders(read_orders('data/orders.csv'))
# ov = overview(orders)
# by_city = analyze_by_city(orders)
# by_cat = analyze_by_category(orders)
# big = analyze_big_orders(orders)
# print('✅ 四个分析函数全部跑通')
# print(f'总览: {ov}')
# print(f'城市排名: {by_city[0]["city"]}排第一')
# print(f'品类排名: {by_cat[0]["category"]}排第一')

In [ ]:
# ============================================================
# 【交付物④】main.py —— 编排全部模块 + 一键出报告
# ============================================================
# 用到的武器：D5-import/__name__、D4-f-string格式化、D3-函数编排
# 
# 小王："最终交付物不是四个函数——是一份能直接拿去开会用的报告。
#        main.py 就是你的总控台：python main.py 一键跑通全流程。"
# 
# 任务：写 main.py 的完整内容
#   1. import 前面写的三个模块
#   2. 按顺序执行：读 → 清洗 → 分析
#   3. 调用 generate_report() 输出格式化的完整报告
#   4. 用 if __name__ == "__main__": 保护入口
#   5. generate_report() 里用 f-string 格式化每一个数字（金额:.2f、百分比:.1%）
# 
# 报告格式要求：
#   ✓ 结论先行——第一段就是核心结论
#   ✓ 每条判断有数字支撑
#   ✓ 大单分析对比大单vs小单的均价差距
#   ✓ 分析产品的成本和售价——计算各品类毛利率
#   ✓ 3条业务建议具体到品类/城市级别
# 
# 👇 在下面写你的 main.py 完整代码：

# ====== main.py ======








# ====== 测试运行 ======
# if __name__ == "__main__":
#     main()

---

## 📝 交付物④续：给mentor小王的正式销售日报

**现在，用你的分析结果，在下面写一份正式的报告。**

格式要求：
- **结论先行**：第一段就是核心结论（"上周整体来看，销售额XX元，XX品类贡献了XX%..."）
- **每条判断配数字**：不是"上海卖得不错"，而是"上海5单贡献了4525元，占总额的64.5%，均单905元远超其他城市"
- **4个维度完整**：总览 → 城市对比 → 品类分析 → 大单分析
- **3条业务建议可落地**：不是说"提升销售额"，而是说"上海客单价高但订单量仅排第二，建议在上海加大数码品类推广力度"
- **结合商品表**：计算各品类的毛利率，在建议中体现利润视角

---

### 上周销售日报（2024-01-08 ~ 2024-01-14）

**致：** 小王（mentor）
**来自：** {你的名字}，数据分析实习生
**日期：** 2024-01-15

---

#### 一、核心结论

{在此写一段3-5句话的总览——数字驱动，结论先行}

...

#### 二、城市对比

{各城市订单数/销售额/均单——从高到低排列}

...

#### 三、品类分析

{各品类订单数/销售额/占比——从高到低排列}

...

#### 四、大单分析

{大单数量/金额/占比/商品分布/大单vs小单均价对比}

...

#### 五、业务建议

1. {建议一——具体到品类或城市，有数字支撑}
2. {建议二——考虑利润维度（参考商品表的成本/售价）}
3. {建议三——可执行、可验证}

...

---

> 💡 **提示**：报告写完后再往下翻。先自己写，参考答案在最后。

---

## 🔀 交付物⑤：Git 操作 + README

小王最后补了一句：**"代码放GitHub上，这是团队标配。"**

### 你的项目现在应该长这样

```
w1_project/
├── data/               # 原始数据（只读！不要手动改）
│   ├── orders.csv
│   ├── users.csv
│   └── products.csv
├── src/                # 源代码
│   ├── __init__.py
│   ├── io.py           # 读数据
│   ├── clean.py        # 洗数据
│   └── analyze.py      # 分析数据
├── output/             # 输出结果
├── main.py             # 入口：python main.py 跑全流程
├── requirements.txt    # 依赖清单
├── README.md           # 项目说明
└── .gitignore          # 告诉Git哪些文件不要管
```

### 在终端执行以下命令（不能在这个Notebook里跑，要开真实终端）

```bash
# 1. 进入项目目录
cd w1_project

# 2. 初始化Git仓库
git init

# 3. 创建 .gitignore（排除不需要版本管理的东西）
echo "venv/" >> .gitignore
echo "__pycache__/" >> .gitignore
echo "*.pyc" >> .gitignore
echo "output/" >> .gitignore

# 4. 添加所有文件
git add .

# 5. 首次提交
git commit -m "W1项目：电商订单分析工具——纯Python实现"

# 6. 在GitHub网页上创建新仓库后，关联远程
git remote add origin https://github.com/你的用户名/w1-order-analysis.git

# 7. 推送
git branch -M main
git push -u origin main

# 8. 在浏览器打开你的仓库地址，确认代码都在上面
```

### README.md 模板（照着写，替换{}里的内容）

```markdown
# 电商订单分析工具

## 项目说明
纯Python实现的订单数据分析工具。读取CSV订单数据，
自动清洗脏数据，输出四维度销售日报。

W1周项目——Python数据分析全量实践。

## 环境配置
```bash
python -m venv venv
# Windows:
venv\Scripts\activate
# Mac/Linux:
source venv/bin/activate
pip install -r requirements.txt
```

## 运行方法
```bash
python main.py
```

## 项目结构
```
├── data/          # 原始CSV数据
├── src/           # 源代码模块
│   ├── io.py      # 数据读取
│   ├── clean.py   # 数据清洗
│   └── analyze.py # 数据分析
├── main.py        # 主入口
└── output/        # 输出结果
```

## 输出说明
运行后控制台输出完整销售日报：
- 总览：订单数 / 销售额 / 平均 / 最高 / 最低
- 城市对比：各城市订单数 / 销售额 / 平均
- 品类分析：各品类订单数 / 销售额 / 占比
- 大单分析：大单数量 / 金额 / 占比 / 分布
- 业务建议：3条数据驱动的可执行建议

## 技术栈
Python 3.x（纯标准库——csv, re, os）
```

---

---

## 📝 自评——诚实面对自己

| 检查项 | ✓/✗ | 备注（卡在哪一步） |
|--------|------|------------------|
| 交付物①：io.py 正确读取三份CSV，金额是int | | |
| 交付物②：clean.py 清洗了"45元"和" 北京 " | | |
| 交付物③-①：overview() 输出正确（含手动找max/min） | | |
| 交付物③-②：analyze_by_city() 按销售额降序 | | |
| 交付物③-③：analyze_by_category() 含占比计算 | | |
| 交付物③-④：analyze_big_orders() 大单vs小单对比 | | |
| 交付物④：main.py 一键跑通，`python main.py` 输出完整报告 | | |
| 交付物⑤：Git仓库创建并push到GitHub | | |
| README让别人能复现你的项目 | | |
| 报告有3条可执行的业务建议（非"提升销售额"废话） | | |
| 你能脱稿讲清楚"从CSV到报告"的完整流程吗？ | | |

---

### 诚实三问

1. **哪些模块你完全独立完成，没有查笔记？**
   
   
2. **哪些你查了笔记或看了参考答案？** → 这些就是你明天的复习重点
   
   
3. **本周D1-D5哪个武器你最不熟练？**
   - [ ] D1 容器（list/dict/set/tuple）
   - [ ] D2 控制流（for/if/while/try）
   - [ ] D3 函数（def/lambda/推导式）
   - [ ] D4 文件IO + 字符串 + 正则
   - [ ] D5 模块化 + 项目结构

---

**明天**：D7复习日——5道闭卷自测，诚实面对薄弱点。你会看到这一周到底学会了多少。

> 今天能独立完成这个项目 → 你有纯Python做数据分析的能力了。
> 下周用pandas重做同样的事，你会发现：「原来3行pandas顶今天30行Python」——但今天写的30行让你**理解**了pandas那3行在做什么。

**这一周你从零到能交付完整分析项目——这是质的飞跃。**

---

## 参考答案（核心代码骨架）

> ⚠️ **先独立完成全部5个交付物再往下翻。** 抄答案不等于学会——自己挣扎出来的才记得住。

---

### src/io.py 参考代码

```python
import csv

def read_orders(path):
    """读取订单CSV，金额转int。脏数据跳过并警告。"""
    orders = []
    with open(path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # 跳过空行
            if not row.get('订单号'):
                continue
            # 尝试转换金额
            try:
                row['金额'] = int(row['金额'])
            except (ValueError, KeyError):
                print(f"⚠️ 警告：{row.get('订单号', '未知')} 金额无法转换 → '{row.get('金额', '缺失')}' → 跳过")
                continue
            orders.append(row)
    return orders

def read_users(path):
    """读取用户CSV"""
    with open(path, 'r', encoding='utf-8') as f:
        return [row for row in csv.DictReader(f) if row.get('用户ID')]

def read_products(path):
    """读取商品CSV，成本和售价转int"""
    products = []
    with open(path, 'r', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if not row.get('商品ID'):
                continue
            row['成本'] = int(row['成本'])
            row['售价'] = int(row['售价'])
            products.append(row)
    return products
```

### src/clean.py 参考代码

```python
import re

def clean_orders(orders):
    """清洗订单：修复金额格式、去城市空格"""
    cleaned = []
    for o in orders:
        # 1. 金额：如果是字符串→提取数字
        if isinstance(o['金额'], str):
            digits = re.sub(r'[^0-9]', '', o['金额'])
            if digits:
                o['金额'] = int(digits)
            else:
                print(f"⚠️ 清洗失败：{o['订单号']} 金额='{o['金额']}' → 跳过")
                continue
        
        # 2. 城市：去掉前后空格
        o['城市'] = o['城市'].strip()
        
        # 3. 检查必填字段
        if not all([o.get('订单号'), o.get('商品'), o.get('品类'), o.get('城市')]):
            print(f"⚠️ {o.get('订单号', '未知')} 必填字段缺失 → 跳过")
            continue
        
        cleaned.append(o)
    return cleaned
```

### src/analyze.py 参考代码

```python
def overview(orders):
    """总览：订单数/销售额/平均/最高/最低"""
    n = len(orders)
    total = sum(o['金额'] for o in orders)
    
    # 手动找最高和最低（培养底层感觉）
    max_amt, max_item = orders[0]['金额'], orders[0]['商品']
    min_amt, min_item = orders[0]['金额'], orders[0]['商品']
    for o in orders:
        if o['金额'] > max_amt:
            max_amt, max_item = o['金额'], o['商品']
        if o['金额'] < min_amt:
            min_amt, min_item = o['金额'], o['商品']
    
    return {
        'count': n,
        'total': total,
        'avg': round(total / n, 2),
        'max': {'amount': max_amt, 'item': max_item},
        'min': {'amount': min_amt, 'item': min_item}
    }

def analyze_by_city(orders):
    """按城市统计——订单数/销售额/平均"""
    stats = {}
    for o in orders:
        c = o['城市']
        if c not in stats:
            stats[c] = {'city': c, 'count': 0, 'total': 0}
        stats[c]['count'] += 1
        stats[c]['total'] += o['金额']
    
    for c in stats:
        stats[c]['avg'] = round(stats[c]['total'] / stats[c]['count'], 2)
    
    # 按销售额降序
    return sorted(stats.values(), key=lambda x: -x['total'])

def analyze_by_category(orders):
    """按品类统计——订单数/销售额/占比"""
    stats = {}
    total_all = sum(o['金额'] for o in orders)
    for o in orders:
        cat = o['品类']
        if cat not in stats:
            stats[cat] = {'category': cat, 'count': 0, 'total': 0}
        stats[cat]['count'] += 1
        stats[cat]['total'] += o['金额']
    
    for cat in stats:
        stats[cat]['pct'] = round(stats[cat]['total'] / total_all * 100, 1)
    
    return sorted(stats.values(), key=lambda x: -x['total'])

def analyze_big_orders(orders, threshold=500):
    """大单分析：>threshold的订单深度分析"""
    big = [o for o in orders if o['金额'] > threshold]
    small = [o for o in orders if o['金额'] <= threshold]
    
    total_all = sum(o['金额'] for o in orders)
    big_total = sum(o['金额'] for o in big)
    small_total = sum(o['金额'] for o in small)
    
    # 大单城市分布
    city_dist = {}
    for o in big:
        c = o['城市']
        city_dist[c] = city_dist.get(c, 0) + 1
    
    return {
        'big_count': len(big),
        'big_total': big_total,
        'count_pct': round(len(big) / len(orders) * 100, 1),
        'amount_pct': round(big_total / total_all * 100, 1),
        'big_items': [o['商品'] for o in big],
        'city_dist': city_dist,
        'big_avg': round(big_total / len(big), 2) if big else 0,
        'small_avg': round(small_total / len(small), 2) if small else 0
    }
```

### main.py 参考代码

```python
from src.io import read_orders, read_users, read_products
from src.clean import clean_orders
from src.analyze import overview, analyze_by_city, analyze_by_category, analyze_big_orders

def generate_report(orders, products):
    """生成格式化的完整销售日报"""
    ov = overview(orders)
    by_city = analyze_by_city(orders)
    by_cat = analyze_by_category(orders)
    big = analyze_big_orders(orders)
    
    # 计算各品类毛利率
    cat_cost = {}
    for p in products:
        cat = p['品类']
        if cat not in cat_cost:
            cat_cost[cat] = {'cost': 0, 'revenue': 0}
        cat_cost[cat]['cost'] += p['成本']
        cat_cost[cat]['revenue'] += p['售价']
    
    print('=' * 60)
    print('📊 速卖通上周销售日报（2024-01-08 ~ 2024-01-14）')
    print('=' * 60)
    print()
    
    # 一、总览
    print('━' * 40)
    print('一、总览')
    print(f'  订单总数：{ov["count"]}笔')
    print(f'  总销售额：{ov["total"]:,}元')
    print(f'  平均每单：{ov["avg"]:,.2f}元')
    print(f'  最高单笔：{ov["max"]["amount"]:,}元（{ov["max"]["item"]}）')
    print(f'  最低单笔：{ov["min"]["amount"]:,}元（{ov["min"]["item"]}）')
    print()
    
    # 二、城市对比
    print('━' * 40)
    print('二、城市对比')
    for c in by_city:
        print(f'  {c["city"]}：{c["count"]}单，{c["total"]:,}元，均{c["avg"]:,.2f}元/单')
    print()
    
    # 三、品类分析
    print('━' * 40)
    print('三、品类分析')
    for cat in by_cat:
        print(f'  {cat["category"]}：{cat["count"]}单，{cat["total"]:,}元（占{cat["pct"]}%）')
    print()
    
    # 四、大单分析
    print('━' * 40)
    print('四、大单分析（>500元）')
    print(f'  大单数量：{big["big_count"]}笔（占{big["count_pct"]}%）')
    print(f'  大单金额：{big["big_total"]:,}元（占{big["amount_pct"]}%）')
    print(f'  大单商品：{"、".join(big["big_items"])}')
    for city, cnt in big['city_dist'].items():
        print(f'  {city}大单：{cnt}笔')
    print(f'  大单均价：{big["big_avg"]:,.2f}元 vs 小单均价：{big["small_avg"]:,.2f}元')
    print()
    
    # 五、业务建议
    print('━' * 40)
    print('五、业务建议')
    print('1. {你的建议1——基于数据的具体判断}')
    print('2. {你的建议2}')
    print('3. {你的建议3}')
    print()
    print('=' * 60)

def main():
    print('🚀 开始分析...')
    orders = read_orders('data/orders.csv')
    users = read_users('data/users.csv')
    products = read_products('data/products.csv')
    print(f'📖 读取：{len(orders)}条订单 | {len(users)}个用户 | {len(products)}个商品')
    
    orders = clean_orders(orders)
    print(f'🧹 清洗后：{len(orders)}条有效订单')
    print()
    
    generate_report(orders, products)

if __name__ == '__main__':
    main()
```

---

### 预期输出验证

运行 `python main.py` 后，确认这些数字大致吻合：

| 指标 | 预期值 | 说明 |
|------|--------|------|
| 有效订单数 | 14 | D010（"待定"）被跳过 |
| 总销售额 | ~7,018 | D010被排除 |
| 平均每单 | ~501.29 | |
| 最高单笔 | 2,599（显示器） | |
| 最低单笔 | 35（跳绳） | D013 "45元" 清洗后=45 |
| 上海销售额 | ~4,525 | 5单（含机械键盘1599+显示器2599） |
| 数码销售额 | ~5,085 | 5单，占总~72% |
| 大单数量 | 3笔 | >500：机械键盘/显示器/跑步鞋 |
| 大单金额占比 | ~69.8% | |

> 如果你的结果跟上面大致对得上（允许四舍五入差异）→ **恭喜，你的分析逻辑是正确的！** 🎉

---

### 偏差诊断

| 如果你的输出/报错是... | 你可能的误解是... | 补救 |
|----------------------|-----------------|------|
| 订单数=15而不是14 | D010"待定"没有被跳过 | 检查 io.py 的 try/except——金额转换失败后应该 continue |
| 金额显示为字符串（如"299"） | CSV读出来数字默认是str | 回到 io.py，确认 `row['金额'] = int(row['金额'])` |
| "45元" 清洗后仍然是 "45元" | clean.py 的 str→int 逻辑没处理这个case | 在clean里检查 `isinstance(o['金额'], str)`，用 `re.sub(r'[^0-9]', '', ...)` |
| 城市=" 北京 "（有空格） | 忘了 .strip() | clean.py: `o['城市'] = o['城市'].strip()` |
| `ModuleNotFoundError: No module named 'src'` | Python找不到src包 | 确认 src/ 目录下有 `__init__.py`；或在同一目录运行 main.py |
| KeyError: '金额' | CSV表头可能有空格或大小写不匹配 | 打印 `row.keys()` 看实际的列名是什么 |
| 大单包含了充电宝(499元) | threshold用错了 | `>500` 不包含500，`>=500` 才包含。按需求用 `>` |

---

**搞定了吗？** 如果5个交付物全部完成 + push到GitHub → 你这一周从零到能交付完整分析项目。这是质的飞跃。